# Google Play Review Ingestion and Database Pipeline — Day 3 Repeated Collection Test

In the previous steps, I connected the Google Play review schema to a working SQLite ingestion pipeline.

Day 1 tested the basic end-to-end flow and immediate duplicate handling. The first controlled run inserted 300 new review records, and the immediate repeated run correctly identified 300 existing duplicate records.

Day 2 continued from the same SQLite database after time had passed. It fetched another 300 reviews and inserted 300 previously unseen review records.

In this Day 3 notebook, I continue from the updated Day 2 SQLite database and run the same ingestion pipeline again.

The goal is to check whether the pipeline remains stable across another collection time and whether it can continue capturing new review records over time.

In [1]:
!pip install -q google-play-scraper pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.1 MB/s eta 0:00:00


## 1. Setup

I first import the packages and set up the folder paths.

For Day 3, the most important point is that I am continuing from the updated database after Day 2. I am not starting from an empty database.

In [2]:
import os
import re
import json
import shutil
import sqlite3
import hashlib
import pandas as pd

from datetime import datetime, timezone
from google_play_scraper import reviews, Sort
from google.colab import files

In [3]:
BASE_DIR = "/content/google_play_ingestion_database_pipeline"

DB_DIR = os.path.join(BASE_DIR, "database")
OUTPUT_DIR = os.path.join(BASE_DIR, "outputs", "run5_ingestion_database_pipeline")

os.makedirs(DB_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

DB_PATH = os.path.join(DB_DIR, "google_play_reviews.sqlite")

print("Base folder:", BASE_DIR)
print("Database path:", DB_PATH)
print("Output folder:", OUTPUT_DIR)

Base folder: /content/google_play_ingestion_database_pipeline
Database path: /content/google_play_ingestion_database_pipeline/database/google_play_reviews.sqlite
Output folder: /content/google_play_ingestion_database_pipeline/outputs/run5_ingestion_database_pipeline


## 2. Load Existing SQLite Database from Day 2

For the Day 3 repeated collection test, I use the updated SQLite database from Day 2.

This is important because the goal is to test whether the pipeline can continue from an existing database across multiple collection times.

I upload the current `google_play_reviews.sqlite` file and copy it to the same database path used by this notebook.

In [4]:
uploaded = files.upload()

uploaded_database_found = False

for filename in uploaded.keys():
    if filename.endswith(".sqlite"):
        source_path = filename
        target_path = DB_PATH
        shutil.copy(source_path, target_path)
        uploaded_database_found = True
        print("Uploaded existing database copied to:", target_path)

if not uploaded_database_found:
    raise ValueError("No .sqlite database file was uploaded. Please upload the updated google_play_reviews.sqlite from Day 2.")

print("Database path used for Day 3:", DB_PATH)

Saving google_play_reviews.sqlite to google_play_reviews.sqlite
Uploaded existing database copied to: /content/google_play_ingestion_database_pipeline/database/google_play_reviews.sqlite
Database path used for Day 3: /content/google_play_ingestion_database_pipeline/database/google_play_reviews.sqlite


## 3. Controlled App Batch

I keep the same controlled app targets and collection settings as Day 1 and Day 2.

Using the same apps and settings makes the repeated collection results easier to compare across days.

In [5]:
APP_TARGETS = [
    {
        "app_name": "YouTube",
        "app_id": "com.google.android.youtube",
        "source_platform": "google_play",
        "country": "us",
        "language": "en"
    },
    {
        "app_name": "TikTok",
        "app_id": "com.zhiliaoapp.musically",
        "source_platform": "google_play",
        "country": "us",
        "language": "en"
    },
    {
        "app_name": "Spotify",
        "app_id": "com.spotify.music",
        "source_platform": "google_play",
        "country": "us",
        "language": "en"
    }
]

COUNT_PER_APP = 100
SORT_ORDER = Sort.NEWEST

app_targets_df = pd.DataFrame(APP_TARGETS)
display(app_targets_df)

,app_name,app_id,source_platform,country,language
0,YouTube,com.google.android.youtube,google_play,us,en
1,TikTok,com.zhiliaoapp.musically,google_play,us,en
2,Spotify,com.spotify.music,google_play,us,en


## 4. Database Schema Check

This cell keeps the database schema available in the notebook.

Because this is a Day 3 continuation, the tables should already exist in the uploaded SQLite database. I still use `CREATE TABLE IF NOT EXISTS` so the notebook can safely connect to the same schema without overwriting existing data.

In [6]:
conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

schema_sql = """
PRAGMA foreign_keys = ON;

CREATE TABLE IF NOT EXISTS app_sources (
    app_source_id INTEGER PRIMARY KEY AUTOINCREMENT,
    source_platform TEXT NOT NULL,
    app_id TEXT NOT NULL,
    app_name TEXT NOT NULL,
    country TEXT NOT NULL,
    language TEXT NOT NULL,
    created_at TEXT NOT NULL,
    UNIQUE(source_platform, app_id, country, language)
);

CREATE TABLE IF NOT EXISTS ingestion_runs (
    run_id TEXT PRIMARY KEY,
    source_platform TEXT NOT NULL,
    run_started_at TEXT NOT NULL,
    run_finished_at TEXT,
    run_status TEXT NOT NULL,
    run_type TEXT NOT NULL,
    scraper_package TEXT,
    sort_order TEXT,
    requested_count_per_app INTEGER,
    notes TEXT
);

CREATE TABLE IF NOT EXISTS ingestion_run_targets (
    run_target_id INTEGER PRIMARY KEY AUTOINCREMENT,
    run_id TEXT NOT NULL,
    app_source_id INTEGER NOT NULL,
    requested_count INTEGER,
    fetched_count INTEGER,
    inserted_new_count INTEGER,
    duplicate_existing_count INTEGER,
    failed_count INTEGER,
    min_review_created_at TEXT,
    max_review_created_at TEXT,
    error_message TEXT,
    created_at TEXT NOT NULL,
    FOREIGN KEY(run_id) REFERENCES ingestion_runs(run_id),
    FOREIGN KEY(app_source_id) REFERENCES app_sources(app_source_id)
);

CREATE TABLE IF NOT EXISTS reviews (
    review_key TEXT PRIMARY KEY,
    app_source_id INTEGER NOT NULL,
    source_review_id TEXT NOT NULL,
    user_name TEXT,
    rating INTEGER,
    thumbs_up_count INTEGER,
    review_created_at TEXT,
    app_version TEXT,
    developer_reply_content TEXT,
    developer_replied_at TEXT,
    raw_response_json TEXT,
    first_seen_run_id TEXT NOT NULL,
    last_seen_run_id TEXT NOT NULL,
    created_at TEXT NOT NULL,
    updated_at TEXT NOT NULL,
    FOREIGN KEY(app_source_id) REFERENCES app_sources(app_source_id),
    FOREIGN KEY(first_seen_run_id) REFERENCES ingestion_runs(run_id),
    FOREIGN KEY(last_seen_run_id) REFERENCES ingestion_runs(run_id),
    UNIQUE(app_source_id, source_review_id)
);

CREATE TABLE IF NOT EXISTS review_texts (
    review_key TEXT PRIMARY KEY,
    raw_text TEXT,
    cleaned_text TEXT,
    raw_text_hash TEXT,
    cleaned_text_hash TEXT,
    created_at TEXT NOT NULL,
    updated_at TEXT NOT NULL,
    FOREIGN KEY(review_key) REFERENCES reviews(review_key)
);

CREATE TABLE IF NOT EXISTS review_quality_flags (
    quality_flag_id INTEGER PRIMARY KEY AUTOINCREMENT,
    run_id TEXT NOT NULL,
    review_key TEXT NOT NULL,
    is_missing_review_id INTEGER NOT NULL,
    is_missing_text INTEGER NOT NULL,
    is_short_text INTEGER NOT NULL,
    is_missing_rating INTEGER NOT NULL,
    is_missing_review_date INTEGER NOT NULL,
    is_repeated_content_in_batch INTEGER NOT NULL,
    content_length INTEGER,
    created_at TEXT NOT NULL,
    FOREIGN KEY(run_id) REFERENCES ingestion_runs(run_id),
    FOREIGN KEY(review_key) REFERENCES reviews(review_key),
    UNIQUE(run_id, review_key)
);
"""

cur.executescript(schema_sql)
conn.commit()

schema_path = os.path.join(OUTPUT_DIR, "schema_used_for_run5.sql")
with open(schema_path, "w", encoding="utf-8") as f:
    f.write(schema_sql)

print("Database connected and schema checked.")
print("Saved schema copy to:", schema_path)

Database connected and schema checked.
Saved schema copy to: /content/google_play_ingestion_database_pipeline/outputs/run5_ingestion_database_pipeline/schema_used_for_run5.sql


## 5. Existing Database Check Before Day 3 Run

Before running the Day 3 collection, I check the existing database contents.

After Day 2, the database should already contain 3 ingestion runs, 9 ingestion run targets, 600 review records, 600 review text records, and 900 quality flag rows.

This check makes sure the Day 3 run is continuing from the updated Day 2 database.

In [7]:
pre_day3_table_counts = []

tables_to_check = [
    "app_sources",
    "ingestion_runs",
    "ingestion_run_targets",
    "reviews",
    "review_texts",
    "review_quality_flags"
]

for table in tables_to_check:
    count = pd.read_sql_query(
        f"SELECT COUNT(*) AS row_count FROM {table};",
        conn
    ).loc[0, "row_count"]

    pre_day3_table_counts.append({
        "table_name": table,
        "row_count_before_day3": count
    })

pre_day3_table_counts_df = pd.DataFrame(pre_day3_table_counts)
display(pre_day3_table_counts_df)

pre_day3_runs = pd.read_sql_query("""
    SELECT
        run_id,
        run_type,
        run_started_at,
        run_finished_at,
        run_status,
        requested_count_per_app
    FROM ingestion_runs
    ORDER BY run_started_at;
""", conn)

display(pre_day3_runs)

existing_runs = pre_day3_table_counts_df.loc[
    pre_day3_table_counts_df["table_name"] == "ingestion_runs",
    "row_count_before_day3"
].iloc[0]

existing_reviews = pre_day3_table_counts_df.loc[
    pre_day3_table_counts_df["table_name"] == "reviews",
    "row_count_before_day3"
].iloc[0]

existing_quality_flags = pre_day3_table_counts_df.loc[
    pre_day3_table_counts_df["table_name"] == "review_quality_flags",
    "row_count_before_day3"
].iloc[0]

if existing_runs < 3 or existing_reviews < 600 or existing_quality_flags < 900:
    raise ValueError("This does not look like the updated Day 2 database. Please upload the Day 2 version of google_play_reviews.sqlite.")

print("Existing database check passed. Day 3 can continue from the updated Day 2 database.")

,table_name,row_count_before_day3
0,app_sources,3
1,ingestion_runs,3
2,ingestion_run_targets,9
3,reviews,600
4,review_texts,600
5,review_quality_flags,900


,run_id,run_type,run_started_at,run_finished_at,run_status,requested_count_per_app
0,run5_20260702_050046,controlled_batch_initial,2026-07-02T05:00:46.526983+00:00,2026-07-02T05:00:47.617467+00:00,completed,100
1,run5_20260702_050059,controlled_batch_duplicate_check,2026-07-02T05:00:59.782587+00:00,2026-07-02T05:01:00.347090+00:00,completed,100
2,run5_day2_20260703_043957,day2_repeated_collection,2026-07-03T04:39:57.579279+00:00,2026-07-03T04:39:58.386382+00:00,completed,100


Existing database check passed. Day 3 can continue from the updated Day 2 database.


## 6. Helper Functions

These helper functions keep the pipeline easier to read.

The cleaning step is intentionally simple. I only trim spaces and normalize repeated whitespace. I do not remove too much content because the raw review text should still be preserved.

In [8]:
def utc_now():
    return datetime.now(timezone.utc).isoformat()


def normalize_datetime(value):
    if value is None:
        return None

    if pd.isna(value):
        return None

    if isinstance(value, datetime):
        if value.tzinfo is None:
            return value.replace(tzinfo=timezone.utc).isoformat()
        return value.astimezone(timezone.utc).isoformat()

    return str(value)


def clean_text(text):
    if text is None:
        return None

    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)

    return text


def hash_text(text):
    if text is None:
        return None

    return hashlib.sha256(str(text).encode("utf-8")).hexdigest()


def make_review_key(app_source_id, source_review_id):
    key_text = f"{app_source_id}|{source_review_id}"
    return hashlib.sha256(key_text.encode("utf-8")).hexdigest()


def get_or_create_app_source(conn, app):
    cur = conn.cursor()
    now = utc_now()

    cur.execute("""
        INSERT OR IGNORE INTO app_sources (
            source_platform,
            app_id,
            app_name,
            country,
            language,
            created_at
        )
        VALUES (?, ?, ?, ?, ?, ?)
    """, (
        app["source_platform"],
        app["app_id"],
        app["app_name"],
        app["country"],
        app["language"],
        now
    ))

    conn.commit()

    cur.execute("""
        SELECT app_source_id
        FROM app_sources
        WHERE source_platform = ?
          AND app_id = ?
          AND country = ?
          AND language = ?
    """, (
        app["source_platform"],
        app["app_id"],
        app["country"],
        app["language"]
    ))

    return cur.fetchone()[0]

## 7. Ingestion Function

This function runs the actual ingestion workflow.

For each app, it does these steps:

1. fetch reviews from Google Play
2. process the raw records
3. insert new reviews into the database
4. update existing reviews if they are already in the database
5. store raw and cleaned text
6. create quality flags
7. save app-level run summary

For Day 3, the most important columns are still `inserted_new_count` and `duplicate_existing_count`.

In [9]:
def run_google_play_ingestion(conn, app_targets, count_per_app, run_type, notes):
    cur = conn.cursor()

    run_id = "run5_day3_" + datetime.now().strftime("%Y%m%d_%H%M%S")
    run_started_at = utc_now()

    cur.execute("""
        INSERT INTO ingestion_runs (
            run_id,
            source_platform,
            run_started_at,
            run_status,
            run_type,
            scraper_package,
            sort_order,
            requested_count_per_app,
            notes
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        run_id,
        "google_play",
        run_started_at,
        "running",
        run_type,
        "google-play-scraper",
        "newest",
        count_per_app,
        notes
    ))

    conn.commit()

    print("Started run:", run_id)

    all_processed_records = []
    target_summaries = []

    for app in app_targets:
        print("\nCollecting:", app["app_name"])

        app_source_id = get_or_create_app_source(conn, app)
        fetched_count = 0
        inserted_new_count = 0
        duplicate_existing_count = 0
        failed_count = 0
        error_message = None

        try:
            raw_reviews, continuation_token = reviews(
                app["app_id"],
                lang=app["language"],
                country=app["country"],
                sort=SORT_ORDER,
                count=count_per_app
            )

            fetched_count = len(raw_reviews)
            processed_records = []

            for raw in raw_reviews:
                original_review_id = raw.get("reviewId")
                is_missing_review_id = 1 if original_review_id is None or str(original_review_id).strip() == "" else 0

                raw_text = raw.get("content")
                cleaned_text = clean_text(raw_text)

                raw_text_hash = hash_text(raw_text)
                cleaned_text_hash = hash_text(cleaned_text)

                if is_missing_review_id == 1:
                    fallback_text = f"{app_source_id}|{raw_text_hash}|{raw.get('at')}|{raw.get('score')}"
                    source_review_id = "missing_review_id_" + hash_text(fallback_text)[:16]
                else:
                    source_review_id = str(original_review_id)

                review_key = make_review_key(app_source_id, source_review_id)

                record = {
                    "run_id": run_id,
                    "app_source_id": app_source_id,
                    "app_name": app["app_name"],
                    "app_id": app["app_id"],
                    "country": app["country"],
                    "language": app["language"],
                    "review_key": review_key,
                    "source_review_id": source_review_id,
                    "is_missing_review_id": is_missing_review_id,
                    "user_name": raw.get("userName"),
                    "rating": raw.get("score"),
                    "thumbs_up_count": raw.get("thumbsUpCount"),
                    "review_created_at": normalize_datetime(raw.get("at")),
                    "app_version": raw.get("reviewCreatedVersion"),
                    "developer_reply_content": raw.get("replyContent"),
                    "developer_replied_at": normalize_datetime(raw.get("repliedAt")),
                    "raw_text": raw_text,
                    "cleaned_text": cleaned_text,
                    "raw_text_hash": raw_text_hash,
                    "cleaned_text_hash": cleaned_text_hash,
                    "raw_response_json": json.dumps(raw, default=str, ensure_ascii=False)
                }

                processed_records.append(record)

            non_missing_hashes = [
                r["cleaned_text_hash"]
                for r in processed_records
                if r["cleaned_text_hash"] is not None
            ]

            hash_counts = pd.Series(non_missing_hashes).value_counts()
            repeated_content_hashes = set(hash_counts[hash_counts > 1].index)

            for record in processed_records:
                now = utc_now()

                cur.execute("""
                    SELECT review_key
                    FROM reviews
                    WHERE review_key = ?
                """, (record["review_key"],))

                existed_before = cur.fetchone() is not None

                if existed_before:
                    duplicate_existing_count += 1
                else:
                    inserted_new_count += 1

                cur.execute("""
                    INSERT INTO reviews (
                        review_key,
                        app_source_id,
                        source_review_id,
                        user_name,
                        rating,
                        thumbs_up_count,
                        review_created_at,
                        app_version,
                        developer_reply_content,
                        developer_replied_at,
                        raw_response_json,
                        first_seen_run_id,
                        last_seen_run_id,
                        created_at,
                        updated_at
                    )
                    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
                    ON CONFLICT(review_key) DO UPDATE SET
                        user_name = excluded.user_name,
                        rating = excluded.rating,
                        thumbs_up_count = excluded.thumbs_up_count,
                        review_created_at = excluded.review_created_at,
                        app_version = excluded.app_version,
                        developer_reply_content = excluded.developer_reply_content,
                        developer_replied_at = excluded.developer_replied_at,
                        raw_response_json = excluded.raw_response_json,
                        last_seen_run_id = excluded.last_seen_run_id,
                        updated_at = excluded.updated_at
                """, (
                    record["review_key"],
                    record["app_source_id"],
                    record["source_review_id"],
                    record["user_name"],
                    record["rating"],
                    record["thumbs_up_count"],
                    record["review_created_at"],
                    record["app_version"],
                    record["developer_reply_content"],
                    record["developer_replied_at"],
                    record["raw_response_json"],
                    run_id,
                    run_id,
                    now,
                    now
                ))

                cur.execute("""
                    INSERT INTO review_texts (
                        review_key,
                        raw_text,
                        cleaned_text,
                        raw_text_hash,
                        cleaned_text_hash,
                        created_at,
                        updated_at
                    )
                    VALUES (?, ?, ?, ?, ?, ?, ?)
                    ON CONFLICT(review_key) DO UPDATE SET
                        raw_text = excluded.raw_text,
                        cleaned_text = excluded.cleaned_text,
                        raw_text_hash = excluded.raw_text_hash,
                        cleaned_text_hash = excluded.cleaned_text_hash,
                        updated_at = excluded.updated_at
                """, (
                    record["review_key"],
                    record["raw_text"],
                    record["cleaned_text"],
                    record["raw_text_hash"],
                    record["cleaned_text_hash"],
                    now,
                    now
                ))

                text = record["cleaned_text"]
                content_length = len(text) if text else 0

                is_missing_text = 1 if text is None or text == "" else 0
                is_short_text = 1 if content_length > 0 and content_length < 5 else 0
                is_missing_rating = 1 if record["rating"] is None else 0
                is_missing_review_date = 1 if record["review_created_at"] is None else 0
                is_repeated_content_in_batch = 1 if record["cleaned_text_hash"] in repeated_content_hashes else 0

                cur.execute("""
                    INSERT OR IGNORE INTO review_quality_flags (
                        run_id,
                        review_key,
                        is_missing_review_id,
                        is_missing_text,
                        is_short_text,
                        is_missing_rating,
                        is_missing_review_date,
                        is_repeated_content_in_batch,
                        content_length,
                        created_at
                    )
                    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
                """, (
                    run_id,
                    record["review_key"],
                    record["is_missing_review_id"],
                    is_missing_text,
                    is_short_text,
                    is_missing_rating,
                    is_missing_review_date,
                    is_repeated_content_in_batch,
                    content_length,
                    now
                ))

            conn.commit()

            all_processed_records.extend(processed_records)

            review_dates = [
                r["review_created_at"]
                for r in processed_records
                if r["review_created_at"] is not None
            ]

            min_review_created_at = min(review_dates) if len(review_dates) > 0 else None
            max_review_created_at = max(review_dates) if len(review_dates) > 0 else None

            print("Fetched:", fetched_count)
            print("Inserted new:", inserted_new_count)
            print("Duplicates:", duplicate_existing_count)

        except Exception as e:
            failed_count = 1
            error_message = str(e)
            min_review_created_at = None
            max_review_created_at = None

            print("Failed:", app["app_name"])
            print("Error:", error_message)

        target_created_at = utc_now()

        cur.execute("""
            INSERT INTO ingestion_run_targets (
                run_id,
                app_source_id,
                requested_count,
                fetched_count,
                inserted_new_count,
                duplicate_existing_count,
                failed_count,
                min_review_created_at,
                max_review_created_at,
                error_message,
                created_at
            )
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, (
            run_id,
            app_source_id,
            count_per_app,
            fetched_count,
            inserted_new_count,
            duplicate_existing_count,
            failed_count,
            min_review_created_at,
            max_review_created_at,
            error_message,
            target_created_at
        ))

        target_summaries.append({
            "run_id": run_id,
            "app_name": app["app_name"],
            "app_id": app["app_id"],
            "requested_count": count_per_app,
            "fetched_count": fetched_count,
            "inserted_new_count": inserted_new_count,
            "duplicate_existing_count": duplicate_existing_count,
            "failed_count": failed_count,
            "min_review_created_at": min_review_created_at,
            "max_review_created_at": max_review_created_at,
            "error_message": error_message
        })

        conn.commit()

    run_finished_at = utc_now()

    cur.execute("""
        UPDATE ingestion_runs
        SET run_finished_at = ?,
            run_status = ?
        WHERE run_id = ?
    """, (
        run_finished_at,
        "completed",
        run_id
    ))

    conn.commit()

    processed_df = pd.DataFrame(all_processed_records)
    target_summary_df = pd.DataFrame(target_summaries)

    print("\nCompleted run:", run_id)

    return run_id, processed_df, target_summary_df

## 8. Day 3 Repeated Collection Run

This is the actual Day 3 repeated collection run.

I use the same app targets and the same collection settings as the previous controlled tests:

- YouTube, TikTok, Spotify
- US / English
- newest reviews
- 100 reviews per app

This run should continue from the updated Day 2 database. If any collected reviews are already in the database, they should be counted as existing duplicates. If new review IDs appear, they should be inserted as new rows.

In [10]:
day3_run_id, day3_records_df, day3_target_summary_df = run_google_play_ingestion(
    conn=conn,
    app_targets=APP_TARGETS,
    count_per_app=COUNT_PER_APP,
    run_type="day3_repeated_collection",
    notes="Day 3 repeated collection test using the existing database to check continued stability and new review capture over time."
)

display(day3_target_summary_df)

day3_records_path = os.path.join(OUTPUT_DIR, f"day3_processed_records_{day3_run_id}.csv")
day3_target_summary_path = os.path.join(OUTPUT_DIR, f"day3_target_summary_{day3_run_id}.csv")

day3_records_df.to_csv(day3_records_path, index=False)
day3_target_summary_df.to_csv(day3_target_summary_path, index=False)

print("Saved Day 3 processed records to:", day3_records_path)
print("Saved Day 3 target summary to:", day3_target_summary_path)

Started run: run5_day3_20260704_032957

Collecting: YouTube
Fetched: 100
Inserted new: 100
Duplicates: 0

Collecting: TikTok
Fetched: 100
Inserted new: 100
Duplicates: 0

Collecting: Spotify
Fetched: 100
Inserted new: 100
Duplicates: 0

Completed run: run5_day3_20260704_032957


,run_id,app_name,app_id,requested_count,fetched_count,inserted_new_count,duplicate_existing_count,failed_count,min_review_created_at,max_review_created_at,error_message
0,run5_day3_20260704_032957,YouTube,com.google.android.youtube,100,100,100,0,0,2026-07-03T01:39:35+00:00,2026-07-03T03:28:03+00:00,None
1,run5_day3_20260704_032957,TikTok,com.zhiliaoapp.musically,100,100,100,0,0,2026-07-02T21:06:36+00:00,2026-07-03T03:24:21+00:00,None
2,run5_day3_20260704_032957,Spotify,com.spotify.music,100,100,100,0,0,2026-07-02T23:23:10+00:00,2026-07-03T03:29:55+00:00,None


Saved Day 3 processed records to: /content/google_play_ingestion_database_pipeline/outputs/run5_ingestion_database_pipeline/day3_processed_records_run5_day3_20260704_032957.csv
Saved Day 3 target summary to: /content/google_play_ingestion_database_pipeline/outputs/run5_ingestion_database_pipeline/day3_target_summary_run5_day3_20260704_032957.csv


## 9. Day 3 Run Comparison

This table compares all ingestion runs so far.

For the Day 3 run, I mainly check:

- how many reviews were fetched
- how many reviews were newly inserted
- how many reviews were already known duplicates
- whether any app-level collection failed
- whether the review date range changed compared with the previous runs

In [11]:
day3_run_comparison = pd.read_sql_query("""
    SELECT
        ir.run_id,
        ir.run_type,
        a.app_name,
        a.app_id,
        a.country,
        a.language,
        irt.requested_count,
        irt.fetched_count,
        irt.inserted_new_count,
        irt.duplicate_existing_count,
        irt.failed_count,
        irt.min_review_created_at,
        irt.max_review_created_at,
        irt.error_message
    FROM ingestion_run_targets irt
    JOIN ingestion_runs ir
        ON irt.run_id = ir.run_id
    JOIN app_sources a
        ON irt.app_source_id = a.app_source_id
    ORDER BY ir.run_started_at, a.app_name;
""", conn)

display(day3_run_comparison)

day3_run_comparison_path = os.path.join(OUTPUT_DIR, "day3_run_comparison.csv")
day3_run_comparison.to_csv(day3_run_comparison_path, index=False)

# Also update the general comparison file.
all_run_comparison_path = os.path.join(OUTPUT_DIR, "all_run_comparison.csv")
day3_run_comparison.to_csv(all_run_comparison_path, index=False)

print("Saved:", day3_run_comparison_path)
print("Updated:", all_run_comparison_path)

,run_id,run_type,app_name,app_id,country,language,requested_count,fetched_count,inserted_new_count,duplicate_existing_count,failed_count,min_review_created_at,max_review_created_at,error_message
0,run5_20260702_050046,controlled_batch_initial,Spotify,com.spotify.music,us,en,100,100,100,0,0,2026-07-01T00:44:56+00:00,2026-07-01T05:00:36+00:00,None
1,run5_20260702_050046,controlled_batch_initial,TikTok,com.zhiliaoapp.musically,us,en,100,100,100,0,0,2026-07-01T00:23:21+00:00,2026-07-01T04:55:51+00:00,None
2,run5_20260702_050046,controlled_batch_initial,YouTube,com.google.android.youtube,us,en,100,100,100,0,0,2026-07-01T04:02:55+00:00,2026-07-01T05:00:41+00:00,None
3,run5_20260702_050059,controlled_batch_duplicate_check,Spotify,com.spotify.music,us,en,100,100,0,100,0,2026-07-01T00:44:56+00:00,2026-07-01T05:00:36+00:00,None
4,run5_20260702_050059,controlled_batch_duplicate_check,TikTok,com.zhiliaoapp.musically,us,en,100,100,0,100,0,2026-07-01T00:23:21+00:00,2026-07-01T04:55:51+00:00,None
5,run5_20260702_050059,controlled_batch_duplicate_check,YouTube,com.google.android.youtube,us,en,100,100,0,100,0,2026-07-01T04:02:55+00:00,2026-07-01T05:00:41+00:00,None
6,run5_day2_20260703_043957,day2_repeated_collection,Spotify,com.spotify.music,us,en,100,100,100,0,0,2026-07-02T00:57:56+00:00,2026-07-02T04:37:07+00:00,None
7,run5_day2_20260703_043957,day2_repeated_collection,TikTok,com.zhiliaoapp.musically,us,en,100,100,100,0,0,2026-07-01T23:08:30+00:00,2026-07-02T04:38:54+00:00,None
8,run5_day2_20260703_043957,day2_repeated_collection,YouTube,com.google.android.youtube,us,en,100,100,100,0,0,2026-07-02T02:53:31+00:00,2026-07-02T04:39:56+00:00,None
9,run5_day3_20260704_032957,day3_repeated_collection,Spotify,com.spotify.music,us,en,100,100,100,0,0,2026-07-02T23:23:10+00:00,2026-07-03T03:29:55+00:00,None


Saved: /content/google_play_ingestion_database_pipeline/outputs/run5_ingestion_database_pipeline/day3_run_comparison.csv
Updated: /content/google_play_ingestion_database_pipeline/outputs/run5_ingestion_database_pipeline/all_run_comparison.csv


## 10. Aggregated Run Summary

This table summarizes each ingestion run at the run level.

It makes it easier to compare the initial controlled run, the immediate duplicate handling test, the Day 2 repeated collection test, and the Day 3 repeated collection test.

In [12]:
aggregated_run_summary = pd.read_sql_query("""
    SELECT
        ir.run_id,
        ir.run_type,
        ir.run_started_at,
        ir.run_finished_at,
        ir.run_status,
        SUM(irt.requested_count) AS total_requested_count,
        SUM(irt.fetched_count) AS total_fetched_count,
        SUM(irt.inserted_new_count) AS total_inserted_new_count,
        SUM(irt.duplicate_existing_count) AS total_duplicate_existing_count,
        SUM(irt.failed_count) AS total_failed_count,
        MIN(irt.min_review_created_at) AS earliest_review_created_at,
        MAX(irt.max_review_created_at) AS latest_review_created_at
    FROM ingestion_runs ir
    JOIN ingestion_run_targets irt
        ON ir.run_id = irt.run_id
    GROUP BY
        ir.run_id,
        ir.run_type,
        ir.run_started_at,
        ir.run_finished_at,
        ir.run_status
    ORDER BY ir.run_started_at;
""", conn)

display(aggregated_run_summary)

aggregated_run_summary_path = os.path.join(OUTPUT_DIR, "aggregated_run_summary.csv")
aggregated_run_summary.to_csv(aggregated_run_summary_path, index=False)

print("Saved:", aggregated_run_summary_path)

,run_id,run_type,run_started_at,run_finished_at,run_status,total_requested_count,total_fetched_count,total_inserted_new_count,total_duplicate_existing_count,total_failed_count,earliest_review_created_at,latest_review_created_at
0,run5_20260702_050046,controlled_batch_initial,2026-07-02T05:00:46.526983+00:00,2026-07-02T05:00:47.617467+00:00,completed,300,300,300,0,0,2026-07-01T00:23:21+00:00,2026-07-01T05:00:41+00:00
1,run5_20260702_050059,controlled_batch_duplicate_check,2026-07-02T05:00:59.782587+00:00,2026-07-02T05:01:00.347090+00:00,completed,300,300,0,300,0,2026-07-01T00:23:21+00:00,2026-07-01T05:00:41+00:00
2,run5_day2_20260703_043957,day2_repeated_collection,2026-07-03T04:39:57.579279+00:00,2026-07-03T04:39:58.386382+00:00,completed,300,300,300,0,0,2026-07-01T23:08:30+00:00,2026-07-02T04:39:56+00:00
3,run5_day3_20260704_032957,day3_repeated_collection,2026-07-04T03:29:57.503814+00:00,2026-07-04T03:29:58.299209+00:00,completed,300,300,300,0,0,2026-07-02T21:06:36+00:00,2026-07-03T03:29:55+00:00


Saved: /content/google_play_ingestion_database_pipeline/outputs/run5_ingestion_database_pipeline/aggregated_run_summary.csv


## 11. Day 3 Database Validation Checks

After the Day 3 run, I run the same database validation checks again.

The most important checks are:

1. there should still be no duplicate review rows for the same app source and source review id
2. every review should still have a linked text record
3. every quality flag should still link back to a review record

In [13]:
day3_validation_checks = []

total_app_sources = pd.read_sql_query("SELECT COUNT(*) AS value FROM app_sources;", conn).loc[0, "value"]
total_runs = pd.read_sql_query("SELECT COUNT(*) AS value FROM ingestion_runs;", conn).loc[0, "value"]
total_targets = pd.read_sql_query("SELECT COUNT(*) AS value FROM ingestion_run_targets;", conn).loc[0, "value"]
total_reviews = pd.read_sql_query("SELECT COUNT(*) AS value FROM reviews;", conn).loc[0, "value"]
total_review_texts = pd.read_sql_query("SELECT COUNT(*) AS value FROM review_texts;", conn).loc[0, "value"]
total_quality_flags = pd.read_sql_query("SELECT COUNT(*) AS value FROM review_quality_flags;", conn).loc[0, "value"]

duplicate_review_rows = pd.read_sql_query("""
    SELECT COUNT(*) AS value
    FROM (
        SELECT app_source_id, source_review_id, COUNT(*) AS row_count
        FROM reviews
        GROUP BY app_source_id, source_review_id
        HAVING COUNT(*) > 1
    );
""", conn).loc[0, "value"]

reviews_without_text_link = pd.read_sql_query("""
    SELECT COUNT(*) AS value
    FROM reviews r
    LEFT JOIN review_texts t
        ON r.review_key = t.review_key
    WHERE t.review_key IS NULL;
""", conn).loc[0, "value"]

quality_flags_without_review = pd.read_sql_query("""
    SELECT COUNT(*) AS value
    FROM review_quality_flags q
    LEFT JOIN reviews r
        ON q.review_key = r.review_key
    WHERE r.review_key IS NULL;
""", conn).loc[0, "value"]

day3_validation_checks.append({"check_name": "total_app_sources", "result_value": total_app_sources})
day3_validation_checks.append({"check_name": "total_ingestion_runs", "result_value": total_runs})
day3_validation_checks.append({"check_name": "total_ingestion_run_targets", "result_value": total_targets})
day3_validation_checks.append({"check_name": "total_reviews", "result_value": total_reviews})
day3_validation_checks.append({"check_name": "total_review_texts", "result_value": total_review_texts})
day3_validation_checks.append({"check_name": "total_quality_flags", "result_value": total_quality_flags})
day3_validation_checks.append({"check_name": "duplicate_review_rows_same_app_source", "result_value": duplicate_review_rows})
day3_validation_checks.append({"check_name": "reviews_without_text_link", "result_value": reviews_without_text_link})
day3_validation_checks.append({"check_name": "quality_flags_without_review", "result_value": quality_flags_without_review})

day3_database_validation_summary = pd.DataFrame(day3_validation_checks)

display(day3_database_validation_summary)

day3_database_validation_summary_path = os.path.join(OUTPUT_DIR, "day3_database_validation_summary.csv")
day3_database_validation_summary.to_csv(day3_database_validation_summary_path, index=False)

database_validation_summary_path = os.path.join(OUTPUT_DIR, "database_validation_summary.csv")
day3_database_validation_summary.to_csv(database_validation_summary_path, index=False)

print("Saved:", day3_database_validation_summary_path)
print("Updated:", database_validation_summary_path)

,check_name,result_value
0,total_app_sources,3
1,total_ingestion_runs,4
2,total_ingestion_run_targets,12
3,total_reviews,900
4,total_review_texts,900
5,total_quality_flags,1200
6,duplicate_review_rows_same_app_source,0
7,reviews_without_text_link,0
8,quality_flags_without_review,0


Saved: /content/google_play_ingestion_database_pipeline/outputs/run5_ingestion_database_pipeline/day3_database_validation_summary.csv
Updated: /content/google_play_ingestion_database_pipeline/outputs/run5_ingestion_database_pipeline/database_validation_summary.csv


## 12. Day 3 Quality Flag Summary

This table checks whether quality flags are still being created and linked correctly after the Day 3 repeated collection run.

In [14]:
day3_quality_flag_summary = pd.read_sql_query("""
    SELECT
        q.run_id,
        ir.run_type,
        COUNT(*) AS total_flag_rows,
        SUM(q.is_missing_review_id) AS missing_review_id_count,
        SUM(q.is_missing_text) AS missing_text_count,
        SUM(q.is_short_text) AS short_text_count,
        SUM(q.is_missing_rating) AS missing_rating_count,
        SUM(q.is_missing_review_date) AS missing_review_date_count,
        SUM(q.is_repeated_content_in_batch) AS repeated_content_in_batch_count,
        AVG(q.content_length) AS avg_content_length
    FROM review_quality_flags q
    JOIN ingestion_runs ir
        ON q.run_id = ir.run_id
    GROUP BY q.run_id, ir.run_type
    ORDER BY ir.run_started_at;
""", conn)

display(day3_quality_flag_summary)

day3_quality_flag_summary_path = os.path.join(OUTPUT_DIR, "day3_quality_flag_summary.csv")
day3_quality_flag_summary.to_csv(day3_quality_flag_summary_path, index=False)

quality_flag_summary_path = os.path.join(OUTPUT_DIR, "quality_flag_summary.csv")
day3_quality_flag_summary.to_csv(quality_flag_summary_path, index=False)

print("Saved:", day3_quality_flag_summary_path)
print("Updated:", quality_flag_summary_path)

,run_id,run_type,total_flag_rows,missing_review_id_count,missing_text_count,short_text_count,missing_rating_count,missing_review_date_count,repeated_content_in_batch_count,avg_content_length
0,run5_20260702_050046,controlled_batch_initial,300,0,0,27,0,0,16,81.476667
1,run5_20260702_050059,controlled_batch_duplicate_check,300,0,0,27,0,0,16,81.476667
2,run5_day2_20260703_043957,day2_repeated_collection,300,0,0,39,0,0,33,81.326667
3,run5_day3_20260704_032957,day3_repeated_collection,300,0,0,30,0,0,30,88.273333


Saved: /content/google_play_ingestion_database_pipeline/outputs/run5_ingestion_database_pipeline/day3_quality_flag_summary.csv
Updated: /content/google_play_ingestion_database_pipeline/outputs/run5_ingestion_database_pipeline/quality_flag_summary.csv


## 13. Raw and Cleaned Text Link Check

This sample confirms that raw review text and cleaned review text are still stored separately and linked through the same review key after the Day 3 run.

In [15]:
day3_sample_reviews = pd.read_sql_query("""
    SELECT
        a.app_name,
        r.source_review_id,
        r.rating,
        r.thumbs_up_count,
        r.review_created_at,
        r.app_version,
        r.first_seen_run_id,
        r.last_seen_run_id,
        t.raw_text,
        t.cleaned_text,
        t.raw_text_hash,
        t.cleaned_text_hash
    FROM reviews r
    JOIN app_sources a
        ON r.app_source_id = a.app_source_id
    JOIN review_texts t
        ON r.review_key = t.review_key
    ORDER BY r.updated_at DESC
    LIMIT 20;
""", conn)

display(day3_sample_reviews)

day3_sample_reviews_path = os.path.join(OUTPUT_DIR, "day3_sample_inserted_reviews.csv")
day3_sample_reviews.to_csv(day3_sample_reviews_path, index=False)

sample_reviews_path = os.path.join(OUTPUT_DIR, "sample_inserted_reviews.csv")
day3_sample_reviews.to_csv(sample_reviews_path, index=False)

print("Saved:", day3_sample_reviews_path)
print("Updated:", sample_reviews_path)

,app_name,source_review_id,rating,thumbs_up_count,review_created_at,app_version,first_seen_run_id,last_seen_run_id,raw_text,cleaned_text,raw_text_hash,cleaned_text_hash
0,Spotify,04116999-01b0-4fac-987c-3851c9703fc0,5,0,2026-07-02T23:23:10+00:00,9.1.60.1970,run5_day3_20260704_032957,run5_day3_20260704_032957,the best,the best,6d766cee0782bfafe7e944af817a3fefa9ce43da13c92f...,6d766cee0782bfafe7e944af817a3fefa9ce43da13c92f...
1,Spotify,b5644198-131c-41dd-b2ab-3cdf121d771c,5,1,2026-07-02T23:24:36+00:00,9.1.60.1970,run5_day3_20260704_032957,run5_day3_20260704_032957,super good 😊😊😊😊😊😊😊 really good song and findin...,super good 😊😊😊😊😊😊😊 really good song and findin...,0f0d2c1ca15d15d9c12a7d2876511bf875cf2c445dda59...,0f0d2c1ca15d15d9c12a7d2876511bf875cf2c445dda59...
2,Spotify,5a19e83b-9607-4cb4-b820-1f3f0b0d6264,5,0,2026-07-02T23:25:59+00:00,9.1.60.1970,run5_day3_20260704_032957,run5_day3_20260704_032957,Very cool music app.,Very cool music app.,a09b22cffd7e525ea5b9bb28a77d9be659311bc62f9d76...,a09b22cffd7e525ea5b9bb28a77d9be659311bc62f9d76...
3,Spotify,ed9a63ba-f1d3-40c7-965d-a937f5c70a59,2,0,2026-07-02T23:26:52+00:00,9.1.60.1970,run5_day3_20260704_032957,run5_day3_20260704_032957,I've been using Spotify for years and the amou...,I've been using Spotify for years and the amou...,1fcd5d71788624a036474527e4e6130abf4b4d7a8f20b7...,1fcd5d71788624a036474527e4e6130abf4b4d7a8f20b7...
4,Spotify,bf618bab-4ba7-492e-9d76-8948705e783e,5,0,2026-07-02T23:30:41+00:00,None,run5_day3_20260704_032957,run5_day3_20260704_032957,only thing I wish this app had is ability to s...,only thing I wish this app had is ability to s...,db4dc60c710816183775f49dfb9b36d18ffb18fda92170...,db4dc60c710816183775f49dfb9b36d18ffb18fda92170...
5,Spotify,f2044d1f-d54f-4f6b-a94f-8d5ae64dae29,5,0,2026-07-02T23:32:42+00:00,9.1.60.1970,run5_day3_20260704_032957,run5_day3_20260704_032957,best music app ever,best music app ever,506cee54b8e751e76d8f2dffaa6bf896800782ed51585c...,506cee54b8e751e76d8f2dffaa6bf896800782ed51585c...
6,Spotify,7a0cde36-b49d-4616-bb9e-859b596baaa6,1,0,2026-07-02T23:34:53+00:00,9.1.60.1970,run5_day3_20260704_032957,run5_day3_20260704_032957,it feels like a pay to win game,it feels like a pay to win game,7197894f8245b8016b650eac28066107f0744f7ff203e2...,7197894f8245b8016b650eac28066107f0744f7ff203e2...
7,Spotify,7bb9af49-018d-40b6-a803-d186a20acea2,4,1,2026-07-02T23:37:42+00:00,9.1.60.1970,run5_day3_20260704_032957,run5_day3_20260704_032957,"If you're using it for music, it's great! but ...","If you're using it for music, it's great! but ...",c5a66e73a6fbdb0128df9dac6453fd79e87966c9126cda...,c5a66e73a6fbdb0128df9dac6453fd79e87966c9126cda...
8,Spotify,845a57cf-516f-4225-b768-ff450f31ae0b,5,0,2026-07-02T23:40:49+00:00,9.1.58.1567,run5_day3_20260704_032957,run5_day3_20260704_032957,love it!,love it!,1545056e30f4c48c8e47c7b34d967997250c37a47c4077...,1545056e30f4c48c8e47c7b34d967997250c37a47c4077...
9,Spotify,ad59cf08-f8de-461f-be0e-2445579957d0,5,0,2026-07-02T23:41:43+00:00,9.1.60.1970,run5_day3_20260704_032957,run5_day3_20260704_032957,good,good,770e607624d689265ca6c44884d0807d9b054d23c473c1...,770e607624d689265ca6c44884d0807d9b054d23c473c1...


Saved: /content/google_play_ingestion_database_pipeline/outputs/run5_ingestion_database_pipeline/day3_sample_inserted_reviews.csv
Updated: /content/google_play_ingestion_database_pipeline/outputs/run5_ingestion_database_pipeline/sample_inserted_reviews.csv


## 14. Day 3 Table Row Counts

This is a quick final check of how many rows are in each table after the Day 3 repeated collection run.

In [16]:
day3_table_counts = []

table_names = [
    "app_sources",
    "ingestion_runs",
    "ingestion_run_targets",
    "reviews",
    "review_texts",
    "review_quality_flags"
]

for table in table_names:
    count = pd.read_sql_query(
        f"SELECT COUNT(*) AS count FROM {table};",
        conn
    ).loc[0, "count"]

    day3_table_counts.append({
        "table_name": table,
        "row_count_after_day3": count
    })

day3_table_counts_df = pd.DataFrame(day3_table_counts)
display(day3_table_counts_df)

day3_table_counts_path = os.path.join(OUTPUT_DIR, "day3_table_counts.csv")
day3_table_counts_df.to_csv(day3_table_counts_path, index=False)

table_counts_path = os.path.join(OUTPUT_DIR, "table_counts.csv")
day3_table_counts_df.to_csv(table_counts_path, index=False)

print("Saved:", day3_table_counts_path)
print("Updated:", table_counts_path)

,table_name,row_count_after_day3
0,app_sources,3
1,ingestion_runs,4
2,ingestion_run_targets,12
3,reviews,900
4,review_texts,900
5,review_quality_flags,1200


Saved: /content/google_play_ingestion_database_pipeline/outputs/run5_ingestion_database_pipeline/day3_table_counts.csv
Updated: /content/google_play_ingestion_database_pipeline/outputs/run5_ingestion_database_pipeline/table_counts.csv


## 15. Day 3 Review-Level Database Export

This file exports the current review-level database after the Day 3 repeated collection test.

The SQLite database is still the main storage, but this CSV is useful for checking and GitHub documentation.

In [17]:
day3_review_level_output = pd.read_sql_query("""
    SELECT
        a.source_platform,
        a.app_name,
        a.app_id,
        a.country,
        a.language,
        r.source_review_id,
        r.rating,
        r.thumbs_up_count,
        r.review_created_at,
        r.app_version,
        r.first_seen_run_id,
        r.last_seen_run_id,
        t.raw_text,
        t.cleaned_text,
        t.raw_text_hash,
        t.cleaned_text_hash
    FROM reviews r
    JOIN app_sources a
        ON r.app_source_id = a.app_source_id
    JOIN review_texts t
        ON r.review_key = t.review_key
    ORDER BY a.app_name, r.review_created_at DESC;
""", conn)

day3_review_level_output_path = os.path.join(OUTPUT_DIR, "day3_review_level_database_export.csv")
day3_review_level_output.to_csv(day3_review_level_output_path, index=False)

review_level_output_path = os.path.join(OUTPUT_DIR, "review_level_database_export.csv")
day3_review_level_output.to_csv(review_level_output_path, index=False)

print("Review-level export shape:", day3_review_level_output.shape)
print("Saved:", day3_review_level_output_path)
print("Updated:", review_level_output_path)

Review-level export shape: (900, 16)
Saved: /content/google_play_ingestion_database_pipeline/outputs/run5_ingestion_database_pipeline/day3_review_level_database_export.csv
Updated: /content/google_play_ingestion_database_pipeline/outputs/run5_ingestion_database_pipeline/review_level_database_export.csv


## 16. Updated Run-Level and Target-Level Exports

I also update the general run-level and app-level output files so the GitHub outputs reflect all runs through Day 3.

In [18]:
ingestion_run_summary = pd.read_sql_query("""
    SELECT
        run_id,
        source_platform,
        run_started_at,
        run_finished_at,
        run_status,
        run_type,
        scraper_package,
        sort_order,
        requested_count_per_app,
        notes
    FROM ingestion_runs
    ORDER BY run_started_at;
""", conn)

display(ingestion_run_summary)

ingestion_run_summary_path = os.path.join(OUTPUT_DIR, "ingestion_run_summary.csv")
ingestion_run_summary.to_csv(ingestion_run_summary_path, index=False)

ingestion_target_summary = pd.read_sql_query("""
    SELECT
        irt.run_id,
        ir.run_type,
        a.app_name,
        a.app_id,
        a.country,
        a.language,
        irt.requested_count,
        irt.fetched_count,
        irt.inserted_new_count,
        irt.duplicate_existing_count,
        irt.failed_count,
        irt.min_review_created_at,
        irt.max_review_created_at,
        irt.error_message
    FROM ingestion_run_targets irt
    JOIN ingestion_runs ir
        ON irt.run_id = ir.run_id
    JOIN app_sources a
        ON irt.app_source_id = a.app_source_id
    ORDER BY ir.run_started_at, a.app_name;
""", conn)

display(ingestion_target_summary)

ingestion_target_summary_path = os.path.join(OUTPUT_DIR, "ingestion_target_summary.csv")
ingestion_target_summary.to_csv(ingestion_target_summary_path, index=False)

duplicate_handling_check_path = os.path.join(OUTPUT_DIR, "duplicate_handling_check.csv")
ingestion_target_summary.to_csv(duplicate_handling_check_path, index=False)

print("Saved:", ingestion_run_summary_path)
print("Saved:", ingestion_target_summary_path)
print("Updated:", duplicate_handling_check_path)

,run_id,source_platform,run_started_at,run_finished_at,run_status,run_type,scraper_package,sort_order,requested_count_per_app,notes
0,run5_20260702_050046,google_play,2026-07-02T05:00:46.526983+00:00,2026-07-02T05:00:47.617467+00:00,completed,controlled_batch_initial,google-play-scraper,newest,100,First controlled ingestion run for database im...
1,run5_20260702_050059,google_play,2026-07-02T05:00:59.782587+00:00,2026-07-02T05:01:00.347090+00:00,completed,controlled_batch_duplicate_check,google-play-scraper,newest,100,Second run using the same targets to test dupl...
2,run5_day2_20260703_043957,google_play,2026-07-03T04:39:57.579279+00:00,2026-07-03T04:39:58.386382+00:00,completed,day2_repeated_collection,google-play-scraper,newest,100,Day 2 repeated collection test using the same ...
3,run5_day3_20260704_032957,google_play,2026-07-04T03:29:57.503814+00:00,2026-07-04T03:29:58.299209+00:00,completed,day3_repeated_collection,google-play-scraper,newest,100,Day 3 repeated collection test using the exist...


,run_id,run_type,app_name,app_id,country,language,requested_count,fetched_count,inserted_new_count,duplicate_existing_count,failed_count,min_review_created_at,max_review_created_at,error_message
0,run5_20260702_050046,controlled_batch_initial,Spotify,com.spotify.music,us,en,100,100,100,0,0,2026-07-01T00:44:56+00:00,2026-07-01T05:00:36+00:00,None
1,run5_20260702_050046,controlled_batch_initial,TikTok,com.zhiliaoapp.musically,us,en,100,100,100,0,0,2026-07-01T00:23:21+00:00,2026-07-01T04:55:51+00:00,None
2,run5_20260702_050046,controlled_batch_initial,YouTube,com.google.android.youtube,us,en,100,100,100,0,0,2026-07-01T04:02:55+00:00,2026-07-01T05:00:41+00:00,None
3,run5_20260702_050059,controlled_batch_duplicate_check,Spotify,com.spotify.music,us,en,100,100,0,100,0,2026-07-01T00:44:56+00:00,2026-07-01T05:00:36+00:00,None
4,run5_20260702_050059,controlled_batch_duplicate_check,TikTok,com.zhiliaoapp.musically,us,en,100,100,0,100,0,2026-07-01T00:23:21+00:00,2026-07-01T04:55:51+00:00,None
5,run5_20260702_050059,controlled_batch_duplicate_check,YouTube,com.google.android.youtube,us,en,100,100,0,100,0,2026-07-01T04:02:55+00:00,2026-07-01T05:00:41+00:00,None
6,run5_day2_20260703_043957,day2_repeated_collection,Spotify,com.spotify.music,us,en,100,100,100,0,0,2026-07-02T00:57:56+00:00,2026-07-02T04:37:07+00:00,None
7,run5_day2_20260703_043957,day2_repeated_collection,TikTok,com.zhiliaoapp.musically,us,en,100,100,100,0,0,2026-07-01T23:08:30+00:00,2026-07-02T04:38:54+00:00,None
8,run5_day2_20260703_043957,day2_repeated_collection,YouTube,com.google.android.youtube,us,en,100,100,100,0,0,2026-07-02T02:53:31+00:00,2026-07-02T04:39:56+00:00,None
9,run5_day3_20260704_032957,day3_repeated_collection,Spotify,com.spotify.music,us,en,100,100,100,0,0,2026-07-02T23:23:10+00:00,2026-07-03T03:29:55+00:00,None


Saved: /content/google_play_ingestion_database_pipeline/outputs/run5_ingestion_database_pipeline/ingestion_run_summary.csv
Saved: /content/google_play_ingestion_database_pipeline/outputs/run5_ingestion_database_pipeline/ingestion_target_summary.csv
Updated: /content/google_play_ingestion_database_pipeline/outputs/run5_ingestion_database_pipeline/duplicate_handling_check.csv


## 17. Create Day 3 Summary Text

This cell creates a short result summary using the actual numbers from the database after the Day 3 run.

I save it as a small markdown file so it can be used later for the README, report, or email update.

In [19]:
day3_total_fetched = int(day3_target_summary_df["fetched_count"].sum())
day3_total_inserted = int(day3_target_summary_df["inserted_new_count"].sum())
day3_total_duplicates = int(day3_target_summary_df["duplicate_existing_count"].sum())
day3_total_failed = int(day3_target_summary_df["failed_count"].sum())

final_total_reviews = int(day3_database_validation_summary.loc[
    day3_database_validation_summary["check_name"] == "total_reviews",
    "result_value"
].iloc[0])

final_total_review_texts = int(day3_database_validation_summary.loc[
    day3_database_validation_summary["check_name"] == "total_review_texts",
    "result_value"
].iloc[0])

final_total_quality_flags = int(day3_database_validation_summary.loc[
    day3_database_validation_summary["check_name"] == "total_quality_flags",
    "result_value"
].iloc[0])

final_duplicate_rows = int(day3_database_validation_summary.loc[
    day3_database_validation_summary["check_name"] == "duplicate_review_rows_same_app_source",
    "result_value"
].iloc[0])

final_missing_text_links = int(day3_database_validation_summary.loc[
    day3_database_validation_summary["check_name"] == "reviews_without_text_link",
    "result_value"
].iloc[0])

final_quality_flags_without_review = int(day3_database_validation_summary.loc[
    day3_database_validation_summary["check_name"] == "quality_flags_without_review",
    "result_value"
].iloc[0])

day3_summary_text = f"""
# Day 3 Repeated Collection Summary

The Day 3 repeated collection run used the existing SQLite database from Day 2.

Day 3 collection results:

- Reviews fetched: {day3_total_fetched}
- New review rows inserted: {day3_total_inserted}
- Existing duplicates identified: {day3_total_duplicates}
- Failed app collections: {day3_total_failed}

After the Day 3 run, the database contained:

- Total unique review records: {final_total_reviews}
- Total linked review text records: {final_total_review_texts}
- Total quality flag rows: {final_total_quality_flags}

Database validation results:

- Duplicate review rows for the same app source and source review ID: {final_duplicate_rows}
- Reviews without linked text records: {final_missing_text_links}
- Quality flags without linked review records: {final_quality_flags_without_review}

This shows whether the pipeline continued to work across another repeated collection time while preserving database integrity, quality flags, and raw/cleaned text linkage.
"""

print(day3_summary_text)

day3_summary_text_path = os.path.join(OUTPUT_DIR, "day3_repeated_collection_summary.md")
with open(day3_summary_text_path, "w", encoding="utf-8") as f:
    f.write(day3_summary_text)

print("Saved:", day3_summary_text_path)


# Day 3 Repeated Collection Summary

The Day 3 repeated collection run used the existing SQLite database from Day 2.

Day 3 collection results:

- Reviews fetched: 300
- New review rows inserted: 300
- Existing duplicates identified: 0
- Failed app collections: 0

After the Day 3 run, the database contained:

- Total unique review records: 900
- Total linked review text records: 900
- Total quality flag rows: 1200

Database validation results:

- Duplicate review rows for the same app source and source review ID: 0
- Reviews without linked text records: 0
- Quality flags without linked review records: 0

This shows whether the pipeline continued to work across another repeated collection time while preserving database integrity, quality flags, and raw/cleaned text linkage.

Saved: /content/google_play_ingestion_database_pipeline/outputs/run5_ingestion_database_pipeline/day3_repeated_collection_summary.md


## 18. Save Updated Day 3 Files

I save the updated database and output files together so they can be uploaded to GitHub.

In [20]:
zip_base_path = os.path.join(BASE_DIR, "run5_ingestion_database_pipeline_day3_files")
zip_path = shutil.make_archive(zip_base_path, "zip", BASE_DIR)

print("Created zip file:", zip_path)

files.download(zip_path)

Created zip file: /content/google_play_ingestion_database_pipeline/run5_ingestion_database_pipeline_day3_files.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Final Conclusion

This Day 3 notebook continues the Google Play ingestion and database pipeline using the updated SQLite database from Day 2.

Before the Day 3 run, the database already contained 3 app sources, 3 ingestion runs, 9 ingestion run targets, 600 unique review records, 600 linked review text records, and 900 quality flag rows. This confirms that the notebook continued from the existing Day 2 database instead of starting from an empty file.

The Day 3 run used the same controlled settings:

- YouTube, TikTok, and Spotify
- US / English
- newest reviews
- 100 reviews per app

The Day 3 repeated collection run fetched 300 reviews across the three apps. Compared with the existing database, all 300 collected records were previously unseen source review IDs. The pipeline inserted 300 new review rows, identified 0 existing duplicates in this run, and had 0 failed app collections.

After the Day 3 run, the database contained 900 unique review records, 900 linked review text records, and 1200 quality flag rows across 4 ingestion runs.

The database validation checks still passed. There were 0 duplicate review rows for the same app source and source review id, 0 reviews without linked text records, and 0 quality flags without linked review records.

This confirms that the pipeline remained stable across another repeated collection time. It also shows that the database-backed workflow can continue from an existing database, capture previously unseen review records over time, preserve ingestion run information, keep quality flags, and maintain the raw/cleaned text linkage.

The next step is to run one more repeated collection test at a later time if needed, or summarize the repeated collection results for John.